In [1]:
import joblib
import copy
import numpy as np
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

try:
    from tabpfn import TabPFNClassifier, TabPFNRegressor
    tabpfn_available = True
except ImportError as e:
    tabpfn_available = False

try:
    from tabicl import TabICLClassifier, TabICLRegressor
    tabicl_available = True
except ImportError as e:
    tabicl_available = False

try:
    from tabfm import TabFMClassifier, TabFMRegressor
    tabfm_available = True
except ImportError as e:
    tabfm_available = False

try:
    from exaonetabular import EXAONETabularClassifier, EXAONETabularRegressor
    tabeo_available = True
except ImportError as e:
    tabeo_available = False

try:
    from deeptab.models import MambularClassifier, MambularRegressor
    from deeptab.models import MambaTabClassifier, MambaTabRegressor
    from deeptab.models import MambAttentionClassifier, MambAttentionRegressor
    from deeptab.models import FTTransformerClassifier, FTTransformerRegressor
    from deeptab.models import AutoIntClassifier, AutoIntRegressor
    from deeptab.models import TabRClassifier, TabRRegressor
    from deeptab.models import TabTransformerClassifier, TabTransformerRegressor
    from deeptab.models import MLPClassifier, MLPRegressor
    deeptab_available = True
except ImportError as e:
    from sklearn.neural_network import MLPClassifier, MLPRegressor
    deeptab_available = False

try:
    from pytabkit import RealMLP_TD_Classifier, RealMLP_TD_Regressor
    from pytabkit import XRFM_D_Regressor, XRFM_D_Classifier
    from pytabkit import TabM_D_Classifier, TabM_D_Regressor
    pytabkit_available = True
except ImportError:
    pytabkit_available = False

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [2]:
%load_ext cuml.accel

In [3]:
model_availability = {
    "tabpfn": tabpfn_available,
    "tabicl": tabicl_available,
    "tabfm": tabfm_available,
    "exaonetabular": tabeo_available,
    "deeptab": deeptab_available,
    "pytabkit": pytabkit_available
}

for package, is_available in model_availability.items():
    print(f"{package}_available = {is_available}")

tabpfn_available = True
tabicl_available = False
tabfm_available = False
exaonetabular_available = False
deeptab_available = False
pytabkit_available = False


In [4]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-09-12-02_55_53_PM'

In [5]:
oof_experiment_config = {
     "experiment": {
        "model": "randomforest",
        "type": "baseline",
        "train_feature_groups": ["ohe/train_baseline.parq", "features/train.parq", "target_encode/train.parq"],
        "test_feature_groups": ["ohe/test_baseline.parq", "features/test.parq", "target_encode/test.parq"],
        "description": "xgboost + classification + baseline + features + TE",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "features": {
        "class_sample_weight": False
    },

    "params": {
        "n_estimators": 1100,
        "max_samples": 0.8,
        "max_depth": 7,
        "max_features": "sqrt",
        "n_jobs": -1
    },
    
    "fit_params": {
    }
}
oof_experiment_config["experiment"]["id"] = f"{dt_str}_{oof_experiment_config["experiment"]["model"]}"
experiment_config = copy.deepcopy(oof_experiment_config)

In [6]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/dataset/data"
    experiments_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/dataset/experiments"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    experiments_path = "../../experiments"
    output_path = "../../"

In [7]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [8]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
X_dfs, X_test_dfs = [], []

for fg in oof_experiment_config["experiment"]["train_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_dfs.append(df)

for fg in oof_experiment_config["experiment"]["test_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_test_dfs.append(df)

X = pd.concat(X_dfs, axis=1)
X_test = pd.concat(X_test_dfs, axis=1)
y = raw_train_df[target_column]

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 42 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               662440 non-null  float64
 1   daily_screen_time_hours           595515 non-null  float64
 2   social_media_hours                557374 non-null  float64
 3   gaming_hours                      564548 non-null  float64
 4   work_study_hours                  639851 non-null  float64
 5   sleep_hours                       646889 non-null  float64
 6   notifications_per_day             623785 non-null  float64
 7   app_opens_per_day                 610659 non-null  float64
 8   weekend_screen_time               579306 non-null  float64
 9   stress_level                      636221 non-null  float64
 10  academic_work_impact              647145 non-null  float64
 11  Female                            691369 non-null  b

In [9]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 42 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               662440 non-null  float64
 1   daily_screen_time_hours           595515 non-null  float64
 2   social_media_hours                557374 non-null  float64
 3   gaming_hours                      564548 non-null  float64
 4   work_study_hours                  639851 non-null  float64
 5   sleep_hours                       646889 non-null  float64
 6   notifications_per_day             623785 non-null  float64
 7   app_opens_per_day                 610659 non-null  float64
 8   weekend_screen_time               579306 non-null  float64
 9   stress_level                      636221 non-null  float64
 10  academic_work_impact              647145 non-null  float64
 11  Female                            691369 non-null  b

In [10]:
def make_model(config):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor), # TODO - fix missing col issue in catboost
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "svm": (SVC, SVR),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor)
    }

    if tabpfn_available:
        model_dict["tabpfn"] = (TabPFNClassifier, TabPFNRegressor)

    if tabicl_available:
        model_dict["tabicl"] = (TabICLClassifier, TabICLRegressor)

    if tabfm_available:
        model_dict["tabfm"] = (TabFMClassifier, TabFMRegressor)

    if tabfm_available:
        model_dict["tabeo"] = (EXAONETabularClassifier, EXAONETabularRegressor)

    if deeptab_available:
        model_dict["mamba"] = (MambularClassifier, MambularRegressor)
        model_dict["mambatab"] = (MambaTabClassifier, MambaTabRegressor)
        model_dict["mambattn"] = (MambAttentionClassifier, MambAttentionRegressor)
        model_dict["ftt"] = (FTTransformerClassifier, FTTransformerRegressor)
        model_dict["autoint"] = (AutoIntClassifier, AutoIntRegressor)
        model_dict["tabr"] = (TabRClassifier, TabRRegressor)
        model_dict["tabt"] = (TabTransformerClassifier, TabTransformerRegressor)

    if pytabkit_available:
        model_dict["realmlp"] = (RealMLP_TD_Classifier, RealMLP_TD_Regressor)
        model_dict["xrfm"] = (XRFM_D_Regressor, XRFM_D_Classifier)
        model_dict["tabm"] = (TabM_D_Classifier, TabM_D_Regressor)

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params)

In [11]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp", "tabpfn"]
    fit_params = config["fit_params"]

    if "features" in config and "class_sample_weight" in config["features"] and config["features"]["class_sample_weight"]:
        train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    else:
        train_sample_weight = None

    if name in no_eval_models:
        model.fit(X_train, y_train, sample_weight=train_sample_weight)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid, sample_weight=train_sample_weight)
    elif name == "lightgbm":
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight, **fit_params)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight)

In [12]:
def predict(config, model, X_feat):
    task = config["experiment"]["target"]
    if task == "classification":
        return model.predict_proba(X_feat)[:, 1]
    else:
        return model.predict(X_feat)

In [13]:
cv_config = experiment_config["cv"]
kf = StratifiedKFold(n_splits=cv_config["n_splits"], random_state=cv_config["random_state"], shuffle=cv_config["shuffle"])

y_cv = pd.Series(index=y.index, dtype=float, name=target_column)
fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = make_model(oof_experiment_config)
    oof_fit(oof_experiment_config, model, X_train, y_train, X_valid, y_valid)

    y_pred = predict(oof_experiment_config, model, X_valid)
    y_cv.iloc[valid_index] = y_pred

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(round(fold_auc_score, 5))

elapsed = time.time() - start_time

y_pred_df = pd.concat([raw_train_id, y_cv], axis=1)

In [14]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

Validation AUC: 0.9499126462315357


In [15]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "train_feature_groups": f"{experiment_config["experiment"]["train_feature_groups"]}",
    "cv": {
        "strategy": f"{experiment_config["cv"]["strategy"]}",
        "n_splits": experiment_config["cv"]["n_splits"],
        "random_state": experiment_config["cv"]["random_state"],
        "fold_scores": fold_scores,
        "mean": round(sum(fold_scores) / len(fold_scores), 5),
        "std": round(float(pd.Series(fold_scores).std(ddof=1)), 5)
    },
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    },
    "training": {
        "duration_seconds": round(elapsed, 2)
    }
}

In [16]:
params = experiment_config["params"]
tree_param_keys = ["n_estimators", "max_iter"]

best_iter = None
if hasattr(model, "best_iteration_"):
    best_iter = model.best_iteration_
elif hasattr(model, "best_iteration"):
    best_iter = model.best_iteration
elif hasattr(model, "n_iter_"):
    n_iter_val = model.n_iter_
    
    if isinstance(n_iter_val, np.ndarray):
        if n_iter_val.size == 1:
            best_iter = n_iter_val.item()
        else:
            best_iter = int(n_iter_val.max())
    else:
        best_iter = int(n_iter_val)

if best_iter is not None:
    for key in tree_param_keys:
        if key in params:
            params[key] = best_iter

training_only_params = [
    "early_stopping_rounds",
    "early_stopping",
    "n_iter_no_change",
    "validation_fraction",
    "eval_set",
]

for key in training_only_params:
    if key in params:
        del params[key]

In [17]:
if experiment_config["features"]["class_sample_weight"]:
    y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)
else:
    y_sample_weight = None

model = make_model(experiment_config)
model.fit(X, y, sample_weight=y_sample_weight)

y_pred = predict(experiment_config, model, X_test)
ss[target_column] = y_pred
ss

,id,addicted_label
0,691369,0.975985
1,691370,0.914892
2,691371,0.873194
3,691372,0.973922
4,691373,0.981678
...,...,...
296297,987666,0.996862
296298,987667,0.779815
296299,987668,0.408896
296300,987669,0.475979


In [18]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

with open(experiment_path / "oof_config.json", "w") as f:
    json.dump(oof_experiment_config, f, indent=4)

with open(experiment_path / "full_config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

y_pred_df.to_csv(experiment_path / "oof.csv", index=False)
joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")
ss.to_csv(experiment_path / f"submission.csv", index=False)